# inplace-op-unsafe-warning — ex1: add_inplace_safe: refuse when .recipe is not None, mutate otherwise

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inplace-op-unsafe-warning`. Running the final beacon cell reports progress against the `Backprop: In-place op unsafe warning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: In-place op unsafe warning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-op-unsafe-warning`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-op-unsafe-warning"
DD_SUBTOPIC = "Backprop: In-place op unsafe warning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## In-place op unsafe — refuse on Recipe-carrying Tensor — quick refresher

In-place ops (`x.add_(y)`, `x.mul_(2)`) overwrite the array in place. If `x` is a non-leaf — i.e. `x.recipe is not None` — its underlying array is **also** stored in some other node's Recipe as a `parent` or `arg`. Mutating it CORRUPTS the graph: a downstream back fn will compute the wrong local gradient because the cached input it reads has been silently replaced.

Canonical guard — raise (or warn) before doing the mutation:

```python
def add_inplace(x: Tensor, y: Tensor) -> Tensor:
    if x.recipe is not None:
        raise RuntimeError(
            'in-place op forbidden on a Tensor with a recipe — '
            'it would corrupt cached values on the graph'
        )
    x.array += y.array
    return x
```

Leaves (`.recipe is None`) are safe to mutate — they're not cached as intermediate values anywhere. Hence the simple rule: **in-place is OK iff `.recipe is None`**. PyTorch's `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation` is the same idea (slightly different scope).

### Exercise 1 — add_inplace_safe: refuse when .recipe is not None, mutate otherwise

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the in-place safety guard: refuse to mutate any Tensor whose Recipe is non-None (would corrupt cached values in the compute graph), allow it only for leaves.
> Keywords: in-place, graph-safety, recipe, guard, mutation
> ```

**KCs targeted:** `inplace-op-unsafe-warning`, `recipe-dataclass`

Implement `add_inplace_safe(x: MiniTensor, y: MiniTensor)` — an in-place `x += y` that **refuses** to mutate `x` when `x` is part of a compute graph.

Rule:

- **If `x.recipe is not None`:** raise `RuntimeError(...)`. The error message must mention 'in-place' (lower-case, no hyphen — the test grep is case-insensitive but unambiguous).
- **Otherwise (`x.recipe is None`):** mutate `x.array` in place (`x.array += y.array`) and return `x`.

**Why the guard exists.** An in-place mutation overwrites `x.array`. If `x` is non-leaf, its array is also stored in other nodes' Recipes as a parent / arg. The next time the reverse pass visits one of those nodes, the back fn reads the *mutated* value instead of the original — silent correctness bug.

Leaves (`.recipe is None`) are safe: they are not cached as intermediates anywhere. Parameter updates in the optimizer (`param.array -= lr * param.grad`) are exactly this case — leaf, no Recipe, mutate freely.

Signature: `add_inplace_safe(x: MiniTensor, y: MiniTensor) -> MiniTensor`. The returned MiniTensor must be the SAME object as `x` (in-place semantics).

In [ ]:
def add_inplace_safe(x: MiniTensor, y: MiniTensor) -> MiniTensor:
    # Guard: any Tensor whose `.array` may be cached in some downstream
    # node's Recipe (i.e. any non-leaf) is unsafe to mutate. The signal
    # is `x.recipe is not None` — leaves never carry a Recipe.
    if x.recipe is not None:
        raise RuntimeError(
            'in-place op forbidden on a Tensor with a recipe — '
            'would corrupt cached values on the graph'
        )
    # Leaf path: safe to mutate.
    x.array += y.array
    return x


<details><summary>Solution</summary>

```python
def add_inplace_safe(x: MiniTensor, y: MiniTensor) -> MiniTensor:
    # Guard: any Tensor whose `.array` may be cached in some downstream
    # node's Recipe (i.e. any non-leaf) is unsafe to mutate. The signal
    # is `x.recipe is not None` — leaves never carry a Recipe.
    if x.recipe is not None:
        raise RuntimeError(
            'in-place op forbidden on a Tensor with a recipe — '
            'would corrupt cached values on the graph'
        )
    # Leaf path: safe to mutate.
    x.array += y.array
    return x
```

**The guard is on `.recipe`, not on `requires_grad`.** A leaf parameter has `requires_grad=True` AND `recipe is None` — and the optimizer mutates it in place every step. Conversely, a non-leaf intermediate could in theory have `requires_grad=False` but still be cached in a Recipe (rare, but possible if the input graph mixed grad-tracked and non-tracked subtrees). The correctness condition is purely about whether the array is cached, which the Recipe presence tracks exactly.

**Why raise, not warn.** A silent warning lets the bug propagate — the user sees slightly-wrong gradients, blames the model, and spends days debugging. Raising forces the user to either (a) realize the in-place was a mistake and replace it with out-of-place, or (b) call `.detach()` first to peel the Recipe off explicitly and own the choice.

**PyTorch's actual message.** `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.` The scope is slightly different (PyTorch tracks leaf-with-rg specifically), but the pattern is identical: refuse the mutation, force the user to be explicit.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()